# Langmuir-probe I–V curve quality analysis

Curated portfolio demonstration of the machine-learning workflow developed for a university computational-physics team project. It combines signal preprocessing, feature engineering, heuristic quality screening, anomaly detection, dimensionality reduction, clustering, and a Random Forest baseline.

**Privacy and data:** this notebook has been stripped of the original outputs, contributor names, shot identifiers, and laboratory measurements. The accompanying sample curves are synthetic and are not experimental results or scientifically validated diagnostics.

Run from the repository root after installing `requirements.txt` and generating the example CSV files with `python scripts/generate_sample_data.py`.

In [ ]:
!pip install -r requirements.txt

In [ ]:
# ============================================================
# MÓDULO ML LISTO PARA COLAB
# Control de calidad de curvas I-V de sonda Langmuir
# Proyecto SCR-1 / Física Computacional
#
# Este bloque:
#   1. Busca automáticamente los archivos sample_curve_*.csv
#      en /content, data y subcarpetas.
#   2. Lee los Excel con engine="calamine" para evitar el error de openpyxl.
#   3. Extrae características físicas/computacionales de cada curva.
#   4. Clasifica curvas como válidas, dudosas o descartables.
#   5. Aplica ML: Isolation Forest, PCA, KMeans y Random Forest.
#   6. Guarda tablas y figuras en /content/resultados_ml_langmuir.
# ============================================================

# ============================================================
# 0. INSTALACIÓN DE LIBRERÍAS NECESARIAS
# ============================================================

!pip install -q python-calamine scikit-learn scipy matplotlib pandas numpy openpyxl

# ============================================================
# 1. IMPORTACIONES
# ============================================================

import os
import re
import glob
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy.stats import skew, kurtosis

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

# ============================================================
# 2. CONFIGURACIÓN GENERAL
# ============================================================

@dataclass
class MLConfig:
    output_dir: str = "results"

    # Rutas donde se buscarán los archivos.
    # Como los subiste manualmente, normalmente estarán en /content
    # o en data.
    search_dirs: tuple = ("data",)

    file_pattern: str = "sample_curve_*.csv"

    # Columnas exactas esperadas según tus archivos
    voltage_exact: str = "voltage_v"
    current_exact: str = "current_a"

    # Palabras clave alternativas por si cambia algún nombre de columna
    voltage_keywords: tuple = (
        "alimentación",
        "alimentacion",
        "volt",
        "voltage",
        "bias",
        "smu",
        "tension",
        "tensión",
    )

    current_keywords: tuple = (
        "corriente",
        "current",
        "amp",
        "ampere",
    )

    # Preprocesamiento
    n_bins: int = 350
    apply_smoothing: bool = True
    savgol_window: int = 21
    savgol_polyorder: int = 3

    # Regiones automáticas por percentiles de voltaje
    ion_percentile: float = 0.20
    electron_percentile: float = 0.80

    # Ventana para buscar región exponencial
    exp_window_fraction: float = 0.25
    min_exp_points: int = 20

    # Machine learning
    contamination: float = 0.20
    random_state: int = 42
    n_clusters: int = 3

    # Criterios físicos preliminares
    te_min_ev: float = 0.1
    te_max_ev: float = 100.0


config = MLConfig()

os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(os.path.join(config.output_dir, "figuras_curvas"), exist_ok=True)
os.makedirs(os.path.join(config.output_dir, "figuras_ml"), exist_ok=True)

# ============================================================
# 3. DEMONSTRATION METADATA
# Synthetic positions only; they are not SCR-1 measurements.
RADIO_MAP = {i: 0.25 + 0.004 * (i - 1) for i in range(1, 17)}
LABEL_MAP = {i: "DEMO" for i in range(1, 17)}

# 4. BÚSQUEDA AUTOMÁTICA DE ARCHIVOS
# ============================================================

def buscar_archivos_curvas(config):
    """
    Busca los archivos sample_curve_*.csv en las rutas configuradas.
    Sirve tanto si están en /content como en data.
    """

    archivos = []

    for carpeta in config.search_dirs:
        if os.path.exists(carpeta):
            patron_directo = os.path.join(carpeta, config.file_pattern)
            patron_recursivo = os.path.join(carpeta, "**", config.file_pattern)

            archivos.extend(glob.glob(patron_directo))
            archivos.extend(glob.glob(patron_recursivo, recursive=True))

    # Eliminar duplicados y ordenar por número de shot
    archivos = sorted(list(set(archivos)), key=lambda p: extraer_shot_id(os.path.basename(p)) or 0)

    # Filtrar solo shots que estén en RADIO_MAP
    archivos_validos = []
    for archivo in archivos:
        shot = extraer_shot_id(os.path.basename(archivo))
        if shot in RADIO_MAP:
            archivos_validos.append(archivo)

    return archivos_validos


def extraer_shot_id(nombre_archivo):
    """
    Extrae el número de shot desde el nombre del archivo.
    Ejemplo:
        sample_curve_001.csv -> 1
    """
    base = os.path.basename(nombre_archivo)
    nums = re.findall(r"\d+", base)

    if len(nums) == 0:
        return None

    return int(nums[-1])


# ============================================================
# 5. LECTURA SEGURA DE EXCEL
# ============================================================

def leer_archivo_curva(path):
    """
    Lee un archivo Excel usando calamine.
    Esto evita el error de openpyxl:
        TypeError: expected <class 'openpyxl.styles.fills.Fill'>
    """

    ext = os.path.splitext(path)[1].lower()

    if ext in [".xlsx", ".xls"]:
        try:
            df = pd.read_excel(path, engine="calamine")
        except Exception as e1:
            try:
                df = pd.read_excel(path, engine="openpyxl")
            except Exception as e2:
                raise RuntimeError(
                    f"No se pudo leer {os.path.basename(path)}.\n"
                    f"Error con calamine: {e1}\n"
                    f"Error con openpyxl: {e2}"
                )

    elif ext == ".csv":
        df = pd.read_csv(path)

    else:
        raise ValueError(f"Formato no soportado: {ext}")

    df = df.dropna(axis=1, how="all")
    df.columns = df.columns.astype(str).str.strip()

    return df


def detectar_columnas_vi(df, config):
    """
    Detecta las columnas de voltaje y corriente.
    Primero intenta con los nombres exactos.
    Luego intenta por palabras clave.
    Finalmente usa las dos primeras columnas numéricas.
    """

    cols = list(df.columns)
    cols_lower = [str(c).lower().strip() for c in cols]

    # 1. Nombres exactos
    if config.voltage_exact in df.columns and config.current_exact in df.columns:
        return config.voltage_exact, config.current_exact

    # 2. Búsqueda por palabra clave
    voltage_col = None
    current_col = None

    for c, cl in zip(cols, cols_lower):
        if any(k.lower() in cl for k in config.voltage_keywords):
            voltage_col = c
            break

    for c, cl in zip(cols, cols_lower):
        if any(k.lower() in cl for k in config.current_keywords):
            if c != voltage_col:
                current_col = c
                break

    if voltage_col is not None and current_col is not None:
        return voltage_col, current_col

    # 3. Convertir columnas a numéricas y usar las dos primeras con datos
    df2 = df.copy()

    for c in df2.columns:
        df2[c] = pd.to_numeric(df2[c], errors="coerce")

    numeric_cols = []

    for c in df2.columns:
        if df2[c].notna().sum() >= 10:
            numeric_cols.append(c)

    if len(numeric_cols) >= 2:
        return numeric_cols[0], numeric_cols[1]

    print("Columnas disponibles:")
    print(df.columns.tolist())

    raise ValueError("No se pudieron detectar columnas de voltaje y corriente.")


# ============================================================
# 6. PREPROCESAMIENTO DE CURVAS
# ============================================================

def limpiar_curva(V, I):
    """
    Limpieza básica:
    - convierte a float,
    - elimina NaN,
    - elimina infinitos,
    - ordena por voltaje.
    """

    V = np.asarray(V, dtype=float)
    I = np.asarray(I, dtype=float)

    mask = np.isfinite(V) & np.isfinite(I)

    V = V[mask]
    I = I[mask]

    if len(V) < 10:
        raise ValueError("Curva con muy pocos puntos válidos.")

    idx = np.argsort(V)

    V = V[idx]
    I = I[idx]

    return V, I


def binning_por_voltaje(V, I, n_bins=350):
    """
    Agrupa la curva en bins de voltaje.
    Usa la mediana de corriente en cada bin para reducir ruido.
    """

    if len(V) < n_bins:
        n_bins = max(20, len(V) // 2)

    bins = np.linspace(np.min(V), np.max(V), n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])

    V_binned = []
    I_binned = []

    for j in range(n_bins):
        mask = (V >= bins[j]) & (V < bins[j + 1])

        if np.sum(mask) >= 1:
            V_binned.append(centers[j])
            I_binned.append(np.median(I[mask]))

    Vb = np.array(V_binned)
    Ib = np.array(I_binned)

    if len(Vb) < 20:
        raise ValueError("Después del binning quedaron muy pocos puntos.")

    return Vb, Ib


def suavizar_corriente(I, config):
    """
    Suavizado Savitzky-Golay.
    Si hay pocos puntos, ajusta automáticamente la ventana.
    """

    if not config.apply_smoothing:
        return I.copy()

    n = len(I)
    window = config.savgol_window

    if window >= n:
        window = n - 1

    if window % 2 == 0:
        window -= 1

    if window < 5:
        return I.copy()

    polyorder = min(config.savgol_polyorder, window - 2)

    try:
        return savgol_filter(I, window_length=window, polyorder=polyorder)
    except Exception:
        return I.copy()


def preparar_curva(path, config):
    """
    Lee, limpia, agrupa y suaviza una curva I-V.
    """

    df = leer_archivo_curva(path)
    v_col, i_col = detectar_columnas_vi(df, config)

    V_raw = pd.to_numeric(df[v_col], errors="coerce").values
    I_raw = pd.to_numeric(df[i_col], errors="coerce").values

    V, I = limpiar_curva(V_raw, I_raw)
    Vb, Ib = binning_por_voltaje(V, I, n_bins=config.n_bins)
    Is = suavizar_corriente(Ib, config)

    return {
        "path": path,
        "archivo": os.path.basename(path),
        "voltage_col": v_col,
        "current_col": i_col,
        "V": Vb,
        "I": Ib,
        "I_smooth": Is,
    }


# ============================================================
# 7. FUNCIONES DE AJUSTE FÍSICO PRELIMINAR
# ============================================================

def r2_score_manual(y, y_pred):
    """
    Calcula R² manualmente.
    """

    y = np.asarray(y)
    y_pred = np.asarray(y_pred)

    mask = np.isfinite(y) & np.isfinite(y_pred)

    y = y[mask]
    y_pred = y_pred[mask]

    if len(y) < 3:
        return np.nan

    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)

    if ss_tot == 0:
        return np.nan

    return 1 - ss_res / ss_tot


def ajuste_lineal(V, I):
    """
    Ajuste lineal I = mV + b.
    """

    V = np.asarray(V, dtype=float)
    I = np.asarray(I, dtype=float)

    mask = np.isfinite(V) & np.isfinite(I)

    V = V[mask]
    I = I[mask]

    if len(V) < 3:
        return np.nan, np.nan, np.nan

    try:
        m, b = np.polyfit(V, I, 1)
        pred = m * V + b
        r2 = r2_score_manual(I, pred)

        return m, b, r2

    except Exception:
        return np.nan, np.nan, np.nan


def estimar_vf(V, I):
    """
    Estima Vf como cruce I(V)=0.
    """

    V = np.asarray(V, dtype=float)
    I = np.asarray(I, dtype=float)

    mask = np.isfinite(V) & np.isfinite(I)

    V = V[mask]
    I = I[mask]

    if len(V) < 3:
        return np.nan, 0

    sign_changes = np.where(np.diff(np.sign(I)) != 0)[0]

    if len(sign_changes) == 0:
        return np.nan, 0

    candidatos = []

    for idx in sign_changes:
        v1, v2 = V[idx], V[idx + 1]
        i1, i2 = I[idx], I[idx + 1]

        if i2 == i1:
            continue

        vf = v1 - i1 * (v2 - v1) / (i2 - i1)
        candidatos.append(vf)

    if len(candidatos) == 0:
        return np.nan, 0

    candidatos = np.array(candidatos)
    centro_v = 0.5 * (np.min(V) + np.max(V))

    vf = candidatos[np.argmin(np.abs(candidatos - centro_v))]

    return vf, len(candidatos)


def buscar_region_exponencial(V, I, Iion_pred, config):
    """
    Busca una región donde ln(I - Iion) sea aproximadamente lineal.
    """

    V = np.asarray(V, dtype=float)
    I = np.asarray(I, dtype=float)
    Iion_pred = np.asarray(Iion_pred, dtype=float)

    Ie = I - Iion_pred

    if not np.any(np.isfinite(Ie)):
        return {
            "m_log": np.nan,
            "b_log": np.nan,
            "r2_log": np.nan,
            "Te_est": np.nan,
            "exp_start": np.nan,
            "exp_stop": np.nan,
            "n_exp_points": 0,
        }

    eps = 1e-30

    try:
        threshold = np.nanpercentile(Ie, 10)
    except Exception:
        threshold = eps

    mask_pos = np.isfinite(Ie) & np.isfinite(V) & (Ie > threshold) & (Ie > eps)

    Vp = V[mask_pos]
    Iep = Ie[mask_pos]

    if len(Vp) < config.min_exp_points:
        return {
            "m_log": np.nan,
            "b_log": np.nan,
            "r2_log": np.nan,
            "Te_est": np.nan,
            "exp_start": np.nan,
            "exp_stop": np.nan,
            "n_exp_points": 0,
        }

    ylog = np.log(Iep)

    window = int(config.exp_window_fraction * len(Vp))
    window = max(config.min_exp_points, window)
    window = min(window, len(Vp))

    best = {
        "r2_log": -np.inf,
        "m_log": np.nan,
        "b_log": np.nan,
        "Te_est": np.nan,
        "exp_start": np.nan,
        "exp_stop": np.nan,
        "n_exp_points": 0,
    }

    for start in range(0, len(Vp) - window + 1):
        stop = start + window

        Vw = Vp[start:stop]
        yw = ylog[start:stop]

        try:
            m, b = np.polyfit(Vw, yw, 1)
            pred = m * Vw + b
            r2 = r2_score_manual(yw, pred)

            if not np.isfinite(m) or m <= 0:
                continue

            Te_est = 1 / m

            if not (config.te_min_ev <= Te_est <= config.te_max_ev):
                continue

            if r2 > best["r2_log"]:
                best = {
                    "r2_log": r2,
                    "m_log": m,
                    "b_log": b,
                    "Te_est": Te_est,
                    "exp_start": Vw[0],
                    "exp_stop": Vw[-1],
                    "n_exp_points": len(Vw),
                }

        except Exception:
            continue

    if best["r2_log"] == -np.inf:
        best["r2_log"] = np.nan

    return best


def estimar_vp_por_derivada(V, I_smooth):
    """
    Estima Vp preliminarmente como el máximo de dI/dV.
    """

    try:
        dIdV = np.gradient(I_smooth, V)

        if not np.any(np.isfinite(dIdV)):
            return np.nan, np.nan

        idx = np.nanargmax(dIdV)

        return V[idx], np.nanmax(dIdV)

    except Exception:
        return np.nan, np.nan


def estimar_vp_por_interseccion(V, m_log, b_log, m_elec, b_elec):
    """
    Estima Vp como intersección aproximada entre la exponencial
    reconstruida y la saturación electrónica lineal.
    """

    if np.any(np.isnan([m_log, b_log, m_elec, b_elec])):
        return np.nan

    try:
        expo_arg = m_log * V + b_log
        expo_arg = np.clip(expo_arg, -700, 700)

        I_exp = np.exp(expo_arg)
        I_sat = m_elec * V + b_elec

        diff = I_exp - I_sat

        if not np.any(np.isfinite(diff)):
            return np.nan

        idx = np.nanargmin(np.abs(diff))

        return V[idx]

    except Exception:
        return np.nan


# ============================================================
# 8. EXTRACCIÓN DE FEATURES
# ============================================================

def extraer_features_de_curva(curva, config):
    """
    Extrae características físicas, estadísticas y computacionales.
    """

    V = curva["V"]
    I = curva["I"]
    Is = curva["I_smooth"]

    archivo = curva["archivo"]
    shot = extraer_shot_id(archivo)

    radio = RADIO_MAP.get(shot, np.nan)
    label = LABEL_MAP.get(shot, "NA")

    v_ion_lim = np.quantile(V, config.ion_percentile)
    v_elec_lim = np.quantile(V, config.electron_percentile)

    mask_ion = V <= v_ion_lim
    mask_elec = V >= v_elec_lim

    m_ion, b_ion, r2_ion = ajuste_lineal(V[mask_ion], Is[mask_ion])
    m_elec, b_elec, r2_elec = ajuste_lineal(V[mask_elec], Is[mask_elec])

    if np.isfinite(m_ion) and np.isfinite(b_ion):
        Iion_pred = m_ion * V + b_ion
    else:
        Iion_pred = np.full_like(V, np.nan)

    exp_info = buscar_region_exponencial(V, Is, Iion_pred, config)

    vf, n_zero_crossings = estimar_vf(V, Is)
    vp_deriv, max_didv = estimar_vp_por_derivada(V, Is)

    vp_inter = estimar_vp_por_interseccion(
        V,
        exp_info["m_log"],
        exp_info["b_log"],
        m_elec,
        b_elec,
    )

    residual = I - Is

    current_range = np.nanmax(I) - np.nanmin(I)

    noise_rms = np.sqrt(np.nanmean(residual ** 2))

    if current_range != 0 and np.isfinite(current_range):
        noise_relative = noise_rms / current_range
    else:
        noise_relative = np.nan

    try:
        dIdV = np.gradient(Is, V)
        d2IdV2 = np.gradient(dIdV, V)
    except Exception:
        dIdV = np.full_like(V, np.nan)
        d2IdV2 = np.full_like(V, np.nan)

    if len(dIdV) > 0:
        monotonic_fraction = np.mean(dIdV > 0)
    else:
        monotonic_fraction = np.nan

    features = {
        "archivo": archivo,
        "shot": shot,
        "label": label,
        "radio_m": radio,

        "n_points": len(V),
        "V_min": np.nanmin(V),
        "V_max": np.nanmax(V),
        "V_range": np.nanmax(V) - np.nanmin(V),

        "I_min": np.nanmin(I),
        "I_max": np.nanmax(I),
        "I_mean": np.nanmean(I),
        "I_std": np.nanstd(I),
        "I_range": current_range,
        "I_abs_max": np.nanmax(np.abs(I)),

        "I_skew": skew(I, nan_policy="omit"),
        "I_kurtosis": kurtosis(I, nan_policy="omit"),

        "noise_rms": noise_rms,
        "noise_relative": noise_relative,

        "dIdV_mean": np.nanmean(dIdV),
        "dIdV_std": np.nanstd(dIdV),
        "dIdV_max": np.nanmax(dIdV),
        "dIdV_min": np.nanmin(dIdV),

        "d2IdV2_mean": np.nanmean(d2IdV2),
        "d2IdV2_std": np.nanstd(d2IdV2),
        "d2IdV2_abs_max": np.nanmax(np.abs(d2IdV2)),

        "monotonic_fraction": monotonic_fraction,

        "m_ion": m_ion,
        "b_ion": b_ion,
        "r2_ion": r2_ion,

        "m_elec": m_elec,
        "b_elec": b_elec,
        "r2_elec": r2_elec,

        "m_log": exp_info["m_log"],
        "b_log": exp_info["b_log"],
        "r2_log": exp_info["r2_log"],
        "Te_est_eV": exp_info["Te_est"],
        "exp_start": exp_info["exp_start"],
        "exp_stop": exp_info["exp_stop"],
        "n_exp_points": exp_info["n_exp_points"],

        "Vf_est": vf,
        "n_zero_crossings": n_zero_crossings,

        "Vp_derivative_est": vp_deriv,
        "max_dIdV": max_didv,
        "Vp_intersection_est": vp_inter,
    }

    return features


# ============================================================
# 9. SCORE DE CALIDAD
# ============================================================

def normalizar_01(x, xmin, xmax, invert=False):
    """
    Normaliza un valor al intervalo [0,1].
    """

    if not np.isfinite(x):
        return 0.0

    if xmax == xmin:
        return 0.0

    z = (x - xmin) / (xmax - xmin)
    z = np.clip(z, 0, 1)

    if invert:
        z = 1 - z

    return float(z)


def calcular_quality_score(row):
    """
    Score físico-computacional de calidad.
    """

    s_r2_ion = normalizar_01(row["r2_ion"], 0.60, 0.98)
    s_r2_elec = normalizar_01(row["r2_elec"], 0.60, 0.98)
    s_r2_log = normalizar_01(row["r2_log"], 0.70, 0.995)

    s_noise = normalizar_01(row["noise_relative"], 0.02, 0.30, invert=True)

    if np.isfinite(row["Vf_est"]) and row["n_zero_crossings"] >= 1:
        s_vf = 1.0
    else:
        s_vf = 0.0

    if np.isfinite(row["Te_est_eV"]) and 0.1 <= row["Te_est_eV"] <= 100:
        s_te = 1.0
    else:
        s_te = 0.0

    s_exp_points = normalizar_01(row["n_exp_points"], 20, 80)

    s_monotonic = normalizar_01(row["monotonic_fraction"], 0.45, 0.90)

    score = (
        0.14 * s_r2_ion +
        0.14 * s_r2_elec +
        0.18 * s_r2_log +
        0.16 * s_noise +
        0.12 * s_vf +
        0.12 * s_te +
        0.08 * s_exp_points +
        0.06 * s_monotonic
    )

    return float(np.clip(score, 0, 1))


def asignar_etiqueta_por_score(score):
    """
    Clasifica por score.
    """

    if score >= 0.70:
        return "valida"

    elif score >= 0.45:
        return "dudosa"

    else:
        return "descartable"


def diagnostico_textual(row):
    """
    Explicación breve de la clasificación.
    """

    razones = []

    if not np.isfinite(row["Vf_est"]):
        razones.append("no presenta cruce claro por cero")

    if np.isfinite(row["noise_relative"]) and row["noise_relative"] > 0.25:
        razones.append("ruido relativo alto")

    if not np.isfinite(row["r2_ion"]) or row["r2_ion"] < 0.70:
        razones.append("saturacion ionica poco lineal")

    if not np.isfinite(row["r2_elec"]) or row["r2_elec"] < 0.70:
        razones.append("saturacion electronica poco lineal")

    if not np.isfinite(row["r2_log"]) or row["r2_log"] < 0.80:
        razones.append("region exponencial debil")

    if not np.isfinite(row["Te_est_eV"]):
        razones.append("no se pudo estimar Te")

    elif row["Te_est_eV"] < 0.1 or row["Te_est_eV"] > 100:
        razones.append("Te fuera de rango fisico preliminar")

    if row["n_exp_points"] < 20:
        razones.append("pocos puntos en region exponencial")

    if len(razones) == 0:
        razones.append("curva con criterios de calidad aceptables")

    return "; ".join(razones)


# ============================================================
# 10. MACHINE LEARNING
# ============================================================

def preparar_matriz_ml(df_features):
    """
    Prepara matriz numérica para ML.
    """

    excluir = [
        "archivo",
        "shot",
        "label",
        "quality_label",
        "diagnostico",
        "ml_outlier_label",
        "cluster_label",
        "rf_pred_label",
    ]

    numeric_cols = []

    for c in df_features.columns:
        if c not in excluir and pd.api.types.is_numeric_dtype(df_features[c]):
            numeric_cols.append(c)

    X = df_features[numeric_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)

    for c in X.columns:
        med = X[c].median()

        if not np.isfinite(med):
            med = 0.0

        X[c] = X[c].fillna(med)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X, X_scaled, numeric_cols, scaler


def ejecutar_ml(df_features, config):
    """
    Ejecuta Isolation Forest, PCA, KMeans y Random Forest.
    """

    df = df_features.copy()

    X, X_scaled, feature_cols, scaler = preparar_matriz_ml(df)

    # Isolation Forest
    contamination = min(config.contamination, 0.5)

    if len(df) <= 5:
        contamination = 0.2

    iso = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=config.random_state,
    )

    iso_pred = iso.fit_predict(X_scaled)
    iso_score = iso.decision_function(X_scaled)

    df["ml_outlier"] = iso_pred
    df["ml_outlier_label"] = np.where(iso_pred == -1, "outlier", "normal")
    df["ml_outlier_score"] = iso_score

    # PCA
    n_components = min(3, X_scaled.shape[0], X_scaled.shape[1])

    pca = PCA(n_components=n_components, random_state=config.random_state)
    X_pca = pca.fit_transform(X_scaled)

    df["PC1"] = X_pca[:, 0]

    if n_components >= 2:
        df["PC2"] = X_pca[:, 1]
    else:
        df["PC2"] = 0.0

    if n_components >= 3:
        df["PC3"] = X_pca[:, 2]
    else:
        df["PC3"] = 0.0

    # KMeans
    n_clusters = min(config.n_clusters, len(df))

    if n_clusters >= 2:
        kmeans = KMeans(
            n_clusters=n_clusters,
            random_state=config.random_state,
            n_init=20,
        )

        clusters = kmeans.fit_predict(X_scaled)

        df["cluster_label"] = clusters

        try:
            sil = silhouette_score(X_scaled, clusters)
        except Exception:
            sil = np.nan

    else:
        kmeans = None
        sil = np.nan
        df["cluster_label"] = 0

    # Random Forest con etiquetas heurísticas
    label_map = {
        "descartable": 0,
        "dudosa": 1,
        "valida": 2,
    }

    y = df["quality_label"].map(label_map).values

    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=config.random_state,
        class_weight="balanced",
    )

    rf.fit(X_scaled, y)

    y_pred = rf.predict(X_scaled)

    df["rf_pred_label_num"] = y_pred

    inv_label_map = {v: k for k, v in label_map.items()}

    df["rf_pred_label"] = pd.Series(y_pred).map(inv_label_map).values

    # Validación cruzada segura
    try:
        counts = pd.Series(y).value_counts()
        min_class = counts.min()

        if min_class >= 2 and len(counts) >= 2:
            cv = StratifiedKFold(
                n_splits=min(3, min_class),
                shuffle=True,
                random_state=config.random_state,
            )

            cv_scores = cross_val_score(rf, X_scaled, y, cv=cv)

            cv_mean = np.mean(cv_scores)
            cv_std = np.std(cv_scores)

        else:
            cv_mean = np.nan
            cv_std = np.nan

    except Exception:
        cv_mean = np.nan
        cv_std = np.nan

    feature_importances = pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": rf.feature_importances_,
        }
    ).sort_values("importance", ascending=False)

    models = {
        "scaler": scaler,
        "isolation_forest": iso,
        "pca": pca,
        "kmeans": kmeans,
        "random_forest": rf,
        "feature_cols": feature_cols,
        "pca_explained_variance_ratio": pca.explained_variance_ratio_,
        "silhouette_score": sil,
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "feature_importances": feature_importances,
    }

    return df, models


# ============================================================
# 11. GRÁFICAS
# ============================================================

def graficar_diagnostico_curva(curva, row, config):
    """
    Gráfica individual de cada curva.
    """

    V = curva["V"]
    I = curva["I"]
    Is = curva["I_smooth"]

    archivo = curva["archivo"]

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.plot(V, I, ".", markersize=3, alpha=0.45, label="Curva binned")
    ax.plot(V, Is, "-", linewidth=2, label="Curva suavizada")

    if np.isfinite(row["exp_start"]) and np.isfinite(row["exp_stop"]):
        ax.axvspan(
            row["exp_start"],
            row["exp_stop"],
            alpha=0.15,
            label="Region exponencial estimada",
        )

    if np.isfinite(row["Vf_est"]):
        ax.axvline(
            row["Vf_est"],
            linestyle="--",
            linewidth=1.5,
            label=f"Vf = {row['Vf_est']:.3g}",
        )

    if np.isfinite(row["Vp_derivative_est"]):
        ax.axvline(
            row["Vp_derivative_est"],
            linestyle=":",
            linewidth=1.5,
            label=f"Vp deriv = {row['Vp_derivative_est']:.3g}",
        )

    titulo = (
        f"{archivo}\n"
        f"Etiqueta: {row['quality_label']} | "
        f"Score: {row['quality_score']:.2f} | "
        f"ML: {row.get('ml_outlier_label', 'NA')}"
    )

    ax.set_title(titulo)
    ax.set_xlabel("Voltaje aplicado V")
    ax.set_ylabel("Corriente I")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

    safe_name = (
        archivo.replace("/", "_")
        .replace("\\", "_")
        .replace(".xlsx", "")
        .replace(".csv", "")
    )

    out = os.path.join(
        config.output_dir,
        "figuras_curvas",
        f"diagnostico_{safe_name}.png",
    )

    plt.tight_layout()
    plt.savefig(out, dpi=200)
    plt.close(fig)


def graficar_pca(df_ml, config):
    """
    PCA 2D.
    """

    fig, ax = plt.subplots(figsize=(8, 6))

    labels = df_ml["quality_label"].unique()

    for lab in labels:
        sub = df_ml[df_ml["quality_label"] == lab]

        ax.scatter(
            sub["PC1"],
            sub["PC2"],
            s=80,
            label=lab,
            alpha=0.8,
        )

        for _, row in sub.iterrows():
            ax.text(
                row["PC1"],
                row["PC2"],
                str(row["shot"]),
                fontsize=8,
            )

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title("PCA de caracteristicas de curvas I-V")
    ax.grid(True, alpha=0.25)
    ax.legend()

    out = os.path.join(config.output_dir, "figuras_ml", "pca_calidad_curvas.png")

    plt.tight_layout()
    plt.savefig(out, dpi=220)
    plt.show()


def graficar_outliers(df_ml, config):
    """
    Gráfica de Isolation Forest.
    """

    df_plot = df_ml.sort_values("ml_outlier_score").copy()

    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(df_plot))

    ax.bar(x, df_plot["ml_outlier_score"].values)

    ax.set_xticks(x)
    ax.set_xticklabels(df_plot["shot"].astype(str).values, rotation=45)

    ax.set_xlabel("Shot")
    ax.set_ylabel("Isolation Forest score")
    ax.set_title("Deteccion de curvas atipicas por ML")
    ax.grid(True, axis="y", alpha=0.25)

    out = os.path.join(
        config.output_dir,
        "figuras_ml",
        "isolation_forest_outliers.png",
    )

    plt.tight_layout()
    plt.savefig(out, dpi=220)
    plt.show()


def graficar_importancias(models, config, top_n=15):
    """
    Importancia de variables.
    """

    imp = models["feature_importances"].head(top_n).copy()
    imp = imp.sort_values("importance")

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.barh(imp["feature"], imp["importance"])

    ax.set_xlabel("Importancia")
    ax.set_title("Variables mas relevantes para clasificar calidad")
    ax.grid(True, axis="x", alpha=0.25)

    out = os.path.join(
        config.output_dir,
        "figuras_ml",
        "feature_importance_random_forest.png",
    )

    plt.tight_layout()
    plt.savefig(out, dpi=220)
    plt.show()


def graficar_score_por_radio(df_ml, config):
    """
    Score de calidad contra radio.
    """

    df_plot = df_ml.dropna(subset=["radio_m"]).sort_values("radio_m")

    if len(df_plot) == 0:
        return

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.scatter(
        df_plot["radio_m"],
        df_plot["quality_score"],
        s=90,
    )

    for _, row in df_plot.iterrows():
        ax.text(
            row["radio_m"],
            row["quality_score"],
            str(row["shot"]),
            fontsize=8,
        )

    ax.set_xlabel("Radio r [m]")
    ax.set_ylabel("Score de calidad")
    ax.set_ylim(-0.05, 1.05)
    ax.set_title("Calidad de curvas I-V segun posicion radial")
    ax.grid(True, alpha=0.25)

    out = os.path.join(
        config.output_dir,
        "figuras_ml",
        "score_calidad_por_radio.png",
    )

    plt.tight_layout()
    plt.savefig(out, dpi=220)
    plt.show()


# ============================================================
# 12. PIPELINE PRINCIPAL
# ============================================================

def ejecutar_pipeline_ml_langmuir(config):
    """
    Ejecuta todo el módulo ML.
    """

    archivos = buscar_archivos_curvas(config)

    print("============================================================")
    print("BUSQUEDA DE ARCHIVOS")
    print("============================================================")
    print(f"Archivos encontrados: {len(archivos)}")

    for archivo in archivos:
        print(archivo)

    if len(archivos) == 0:
        raise FileNotFoundError(
            "No se encontraron archivos sample_curve_*.csv.\n"
            "Revisa que estén en /content o data."
        )

    curvas = []
    rows = []
    errores = []

    print("\n============================================================")
    print("PROCESAMIENTO DE CURVAS")
    print("============================================================")

    for path in archivos:
        try:
            curva = preparar_curva(path, config)
            features = extraer_features_de_curva(curva, config)

            curvas.append(curva)
            rows.append(features)

            print(f"OK: {os.path.basename(path)}")

        except Exception as e:
            print(f"ERROR: {os.path.basename(path)} -> {e}")

            errores.append(
                {
                    "archivo": os.path.basename(path),
                    "error": str(e),
                }
            )

    if len(rows) == 0:
        raise RuntimeError("No se pudo procesar ninguna curva.")

    df_features = pd.DataFrame(rows)

    df_features["quality_score"] = df_features.apply(calcular_quality_score, axis=1)
    df_features["quality_label"] = df_features["quality_score"].apply(asignar_etiqueta_por_score)
    df_features["diagnostico"] = df_features.apply(diagnostico_textual, axis=1)

    df_ml, models = ejecutar_ml(df_features, config)

    features_path = os.path.join(config.output_dir, "features_curvas_iv.csv")
    ml_path = os.path.join(config.output_dir, "clasificacion_ml_curvas_iv.csv")
    imp_path = os.path.join(config.output_dir, "importancia_variables_ml.csv")
    err_path = os.path.join(config.output_dir, "errores_procesamiento.csv")

    df_features.to_csv(features_path, index=False)
    df_ml.to_csv(ml_path, index=False)
    models["feature_importances"].to_csv(imp_path, index=False)

    if len(errores) > 0:
        pd.DataFrame(errores).to_csv(err_path, index=False)

    df_by_file = {row["archivo"]: row for _, row in df_ml.iterrows()}

    for curva in curvas:
        row = df_by_file[curva["archivo"]]
        graficar_diagnostico_curva(curva, row, config)

    graficar_pca(df_ml, config)
    graficar_outliers(df_ml, config)
    graficar_importancias(models, config)
    graficar_score_por_radio(df_ml, config)

    print("\n============================================================")
    print("RESUMEN DEL MODULO ML")
    print("============================================================")

    print("\nClasificacion por score:")
    print(df_ml["quality_label"].value_counts())

    print("\nOutliers detectados por Isolation Forest:")
    columnas_outliers = [
        "archivo",
        "shot",
        "label",
        "radio_m",
        "quality_score",
        "ml_outlier_label",
        "diagnostico",
    ]

    print(
        df_ml[df_ml["ml_outlier_label"] == "outlier"][columnas_outliers]
        .to_string(index=False)
    )

    print("\nVarianza explicada por PCA:")
    print(models["pca_explained_variance_ratio"])

    print("\nSilhouette score de KMeans:")
    print(models["silhouette_score"])

    print("\nValidacion cruzada RF sobre etiquetas heuristicas:")
    print(f"Accuracy media = {models['cv_mean']}, std = {models['cv_std']}")

    print("\nArchivos guardados en:")
    print(config.output_dir)

    print("\nTabla principal:")
    columnas_resumen = [
        "archivo",
        "shot",
        "label",
        "radio_m",
        "quality_score",
        "quality_label",
        "ml_outlier_label",
        "Te_est_eV",
        "Vf_est",
        "Vp_derivative_est",
        "r2_ion",
        "r2_log",
        "r2_elec",
        "noise_relative",
        "diagnostico",
    ]

    print(df_ml[columnas_resumen].to_string(index=False))

    return df_ml, models, curvas, errores


# ============================================================
# 13. EJECUCIÓN FINAL
# ============================================================

df_ml, models_ml, curvas_ml, errores_ml = ejecutar_pipeline_ml_langmuir(config)

df_ml.head()